# See, Detect, Understand: Computer Vision introduction (by Statistics and Data Science Society)

## PARTICIPANT TEMPLATE

> **What you are going to do:** You are prototyping photo tags for a photo collection. Your prototype should suggest an ImageNet category for a photograph and display the model's scores. You will also investigate when those suggestions are unreliable.

### Learning goals

By the end of this introductory example, you should be able to:

- Read an image's shape and explain its RGB pixel values.
- Prepare an image for an existing CNN and run a prediction.
- Explain the difference between inference, training and transfer learning.
- Interpret model scores and identify limitations of a fixed category list.

### Workshop route

| Section | Activity | Suggested use |
| --- | --- | --- |
| 0. Setup | Load the tools and a sample image | Provided code |
| 1. Image representation | Inspect numbers and RGB channels | Exercise 1 |
| 2. Image classification | Resize, batch, preprocess and predict | Exercises 2-5 |
| 3. Investigating mistakes | Compare original and blurred images | Optional discussion |
| 4. Your turn | Investigate one new image and explain its results | Optional short challenge |
| 5. Features and CNN intuition | Compare a fixed edge filter with learned activations | Optional presenter extension |


### How to use this template

1. Open [Google Colab](https://colab.research.google.com/) and choose **File > Upload notebook**.
2. Connect to a Python runtime. A default CPU runtime is enough for this small inference example.
3. Run code cells in order using the play button or **Shift+Enter**.
4. Replace the `None` values marked `TODO` in Exercises 1-5 with your code. An unfinished exercise stops with a message asking you to complete it.

The first run downloads a sample photo, model weights and the ImageNet label lookup. Internet access is required.

# Section 0: Setup and sample image

The libraries have different jobs: **NumPy** stores numbers, **Pillow** opens images, **Matplotlib** displays them, and **TensorFlow/Keras** runs the model. `import` makes these tools available to Python.

Colab normally includes these packages. Run this cell first. If an import fails, try a fresh standard runtime before changing packages.

In [ ]:
import io
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image, ImageOps, ImageFilter, UnidentifiedImageError
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)

## A common example for everyone

Keep `image_source = "sample"` for your first run. Everyone will use the same Labrador photograph from a TensorFlow tutorial. Later, change it to `"upload"` to select one JPEG or PNG from your computer.

The supplied code handles downloading and file opening so you can focus on the vision concepts. `convert("RGB")` ensures the image has three colour channels. `exif_transpose` respects orientation information from a camera.

**Image credit:** *YellowLabradorLooking new.jpg*, original photo by Elf, derivative/retouch by Djmirko and FT2, via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:YellowLabradorLooking_new.jpg), [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/). The image transformations shown in this notebook include resizing, channel views, blur and an edge view. The transformed sample image views retain this image credit and licence.

In [ ]:
image_source = "sample"  # Change to "upload" when you want to try your own photo.

if image_source == "sample":
    image_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/YellowLabradorLooking_new.jpg"
    image_path = tf.keras.utils.get_file("YellowLabradorLooking_new.jpg", origin=image_url)
    with Image.open(image_path) as image_file:
        img = ImageOps.exif_transpose(image_file).convert("RGB")
    image_name = "Sample Labrador photograph"
elif image_source == "upload":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Rerun this cell and upload exactly one JPEG or PNG image.")
    image_name = next(iter(uploaded))
    try:
        with Image.open(io.BytesIO(uploaded[image_name])) as image_file:
            img = ImageOps.exif_transpose(image_file).convert("RGB")
    except UnidentifiedImageError as exc:
        raise ValueError("Try a JPEG or PNG. Export HEIC phone photos to JPEG first.") from exc
else:
    raise ValueError('image_source must be "sample" or "upload".')

plt.figure(figsize=(6, 4))
plt.imshow(img)
plt.title(image_name)
plt.axis("off")
plt.show()

# Section 1: How does a computer see?

> **Guiding question 1:** What numbers represent this photograph?

An RGB image has shape **(height, width, 3)**. Each pixel contains red, green and blue channel values. In this 8-bit representation, each value ranges from 0 to 255.

**Exercise 1:** Convert `img` to a NumPy array named `pixels`. Then read the shape and the top-left pixel.

<details><summary>Hint</summary>
NumPy's `asarray` function converts an image into an array. Python starts indexing from zero, so `pixels[0, 0]` selects the first row and first column.
</details>

In [ ]:
# TODO 1: Replace None with a NumPy array made from img.
pixels = None
if pixels is None:
    raise NotImplementedError("Complete Exercise 1 before continuing.")

print("Shape (height, width, channels):", pixels.shape)
print("First pixel [R, G, B]:", pixels[0, 0]) #the three channel values for the first pixel (most topleft)
print("Red-channel values in a 5x5 patch:") #only the red channel values in the 25 pixels at the top left
print(pixels[:5, :5, 0])

## RGB channels

The supplied cell shows the three channels as intensity plots. All panels use the same 0-255 range. These are three measurements at each location in the same photograph.

**Pause and discuss:** What colour does `[255, 0, 0]` represent? What about `[0, 0, 0]` and `[255, 255, 255]`?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for channel, (name, colour_map) in enumerate(
    [("Red", "Reds"), ("Green", "Greens"), ("Blue", "Blues")]
):
    axes[channel].imshow(pixels[:, :, channel], cmap=colour_map, vmin=0, vmax=255)
    axes[channel].set_title(name + " channel intensity")
    axes[channel].axis("off")
plt.tight_layout()
plt.show()
#shows what happens if you only look at one particular RGB channel of all pixels

## Vocabulary checkpoint

| Term | Meaning in this example |
| --- | --- |
| Pixel | Values at one location in an image |
| Channel | One measurement per location, such as red intensity |
| Shape | The dimensions of an array |
| Batch | A collection of examples that a model processes together |
| Class or label | One of the model's possible output categories |
| Feature | Information that helps a model make a prediction |

An image's dimensions describe the input. They do not tell us its semantic category. We need a model to connect visual patterns to categories.

# Section 2: Image classification with a pretrained CNN

> **Guiding question 2:** How can we turn these numbers into a label for the photo?

MobileNetV2 is a **Convolutional Neural Network**, or CNN. Its convolutional layers use learned filters to respond to local patterns. Later layers combine earlier representations into information useful for classification.

`weights="imagenet"` loads parameters learned from ImageNet. We keep the existing classifier, which scores 1,000 ImageNet categories. The supplied cell loads the model once.

| Process | What happens |
| --- | --- |
| Training | Labelled examples and a loss guide changes to learned parameters |
| Inference | An existing model uses its learned parameters to make predictions |
| Transfer learning | We reuse a pretrained representation and train a classifier for a new task, with optional further fine-tuning |

This notebook demonstrates **inference**. Simply uploading a photo does not train or adapt the model.

In [ ]:
model = MobileNetV2(weights="imagenet", include_top=True)
print("Model input shape:", model.input_shape)
print("Model output shape:", model.output_shape)
#the first value of output_shape means the number of images in one batch, while the next one refers to how many categories to which the model assigns score

## Exercise 2: Resize the image

Our default classifier expects a **224 x 224 RGB image**.

Replace `None` with the target size tuple. Then compare the original and resized shapes. Simple resizing may stretch a photo; we use it here to keep the first pipeline easy to follow.

<details><summary>Hint</summary>
Pillow's `resize` takes a `(width, height)` tuple. Both values should be 224 in this example.
</details>

In [ ]:
# TODO 2: Replace None with the target (width, height).
target_size = None
if target_size is None:
    raise NotImplementedError("Complete Exercise 2 before continuing.")

model_img = img.resize(target_size, Image.Resampling.BILINEAR)
image_array = np.asarray(model_img, dtype=np.float32)
print("Original shape:", pixels.shape)
print("Resized shape:", image_array.shape)
assert image_array.shape == (224, 224, 3), "Check the target size and RGB channels."

## Exercise 3: Add a batch dimension

The model expects **(batch size, height, width, channels)**. For one photo, this is `(1, 224, 224, 3)`.

Replace `None` with code that adds a new dimension at axis 0.

<details><summary>Hint</summary>
Use NumPy's `expand_dims` function on `image_array`, with `axis=0`. This groups the image into a batch containing one example.
</details>

In [ ]:
# TODO 3: Add a dimension at axis 0 to image_array.
batch = None
if batch is None:
    raise NotImplementedError("Complete Exercise 3 before continuing.")

print("Batch shape:", batch.shape)
assert batch.shape == (1, 224, 224, 3), "The first dimension should count one image."

## Exercise 4: Apply model-specific preprocessing

MobileNetV2 expects values scaled from `[0, 255]` into `[-1, 1]`. Call its `preprocess_input` function on a copy of `batch` to create `x`.

Do not divide by 255 first. MobileNetV2's preprocessing already performs the required scaling. Another model may use another convention.

<details><summary>Hint</summary>
`preprocess_input` is already imported from MobileNetV2. Pass it `batch.copy()` so the original 0-255 values remain available for comparison.
</details>

In [ ]:
# TODO 4: Apply preprocess_input to a copy of batch.
x = None
if x is None:
    raise NotImplementedError("Complete Exercise 4 before continuing.")

print("Original first pixel:", batch[0, 0, 0])
print("Preprocessed first pixel:", x[0, 0, 0])
print("Input shape:", x.shape)
print("Input range:", float(x.min()), "to", float(x.max()))
assert np.allclose(x, batch / 127.5 - 1.0), "Check model-specific scaling."

## Exercise 5: Run the classifier

Call the model's `predict` method on `x`. The supplied decoder converts category indices into readable labels and shows the highest-scoring categories.

<details><summary>Hint</summary>
Use `model.predict(x, verbose=0)`. `verbose=0` hides the progress bar. This method makes predictions without a training step.
</details>

In [ ]:
# TODO 5: Predict on x using the loaded model.
predictions = None
if predictions is None:
    raise NotImplementedError("Complete Exercise 5 before continuing.")

top_k = 3  # Try 5 after your first successful run.
top_predictions = decode_predictions(predictions, top=top_k)[0]
print("Prediction shape:", predictions.shape)
for rank, (_, label, score) in enumerate(top_predictions, start=1):
    print(f"{rank}. {label.replace('_', ' ')}: {score:.1%}")

## Interpreting the result

> **Guiding question 3:** What do the labels and scores tell us, and what can we conclude?

The classifier scores the **whole image**. ImageNet includes detailed categories, such as particular dog breeds. A category may be more specific than the broad label you had in mind.

The default final layer uses **softmax** (a function turning a vector of raw, unnormalized numbers into a clean probability distribution), so all 1,000 scores sum to approximately 1. The top three alone need not sum to 100%.

**Discussion questions**

1. Is the highest-scoring label a reasonable description of the photograph?
2. Is a score of 90% the same as 90% test accuracy?
3. If two objects appear in a photo, will this classifier locate both of them?
4. What happens if the true subject has no matching ImageNet category?

Write your interpretation in the answer cell below. Look at the photo and the printed results before making a claim.

In [ ]:
labels = [label.replace("_", " ") for _, label, _ in top_predictions]
scores = [float(score) * 100 for _, _, score in top_predictions]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(model_img)
axes[0].set_title("Image supplied to the model")
axes[0].axis("off")
axes[1].barh(labels, scores, color="#3767a6")
axes[1].invert_yaxis()
axes[1].set_xlim(0, 100)
axes[1].set_xlabel("Model score (%)")
axes[1].set_title("Highest-scoring ImageNet categories")
plt.tight_layout()
plt.show()

print("Sum over all categories:", float(predictions.sum()))

### Your interpretation

- Highest-scoring label and score: ...
- Why the label seems reasonable or unreasonable: ...
- One limitation of the result: ...

# Section 3: Investigating mistakes

> **Guiding question 4:** How does a change in the image affect the model's suggestion?

The following supplied code compares an original image, a moderately blurred image and a strongly blurred image. It always applies blur **after resizing**, so the blur radius has a comparable meaning.

Before running it, predict whether the highest-scoring label will change. After running it, compare the observed results with your prediction. Scores may increase or decrease. Keep the result as observed.

Changing an image for this experiment is a transformation at inference time. Data augmentation uses transformations to create additional training examples during training.

In [ ]:
blur_radii = [0, 4, 12]  # Try a different positive radius after the first comparison.
results = []
fig, axes = plt.subplots(1, len(blur_radii), figsize=(12, 4))

for axis, radius in zip(axes, blur_radii):
    changed_img = model_img.filter(ImageFilter.GaussianBlur(radius=radius))
    changed_batch = np.expand_dims(np.asarray(changed_img, dtype=np.float32), axis=0)
    changed_x = preprocess_input(changed_batch.copy())
    changed_predictions = model.predict(changed_x, verbose=0)
    _, label, score = decode_predictions(changed_predictions, top=1)[0][0]
    results.append({"blur_radius": radius, "label": label, "score": float(score)})
    axis.imshow(changed_img)
    axis.set_title(f"Blur radius {radius}\n{label.replace('_', ' ')} ({score:.1%})", fontsize=11)
    axis.axis("off")

plt.tight_layout()
plt.show()
for result in results:
    print(f"Radius {result['blur_radius']:>2}: {result['label']:<24} {result['score']:.1%}")

### Observation questions

- Did the highest-scoring label change? At which blur radius?
- Did its score increase or decrease?
- Which details disappeared from the photograph?
- Can this experiment on one photo establish that the model is reliable across a campus photo collection? What further evidence would you need?

**Your observations:** ...

# Section 4: Your turn

Choose **one question** below and investigate it with a new image. Change `image_source` to `"upload"`, rerun the image-loading cell, then rerun Sections 1 and 2. Keep the already-loaded model. Rerun Section 3 if your question involves blur.

- Does the model suggest a sensible label for a clear photo of an everyday object?
- How does it respond to a cluttered scene containing several objects?
- How does it respond to an SDS logo or another subject outside its existing categories?

### Your task

1. State your question and your expected outcome.
2. Run the code and record the actual highest-scoring label and score.
3. Describe whether the suggestion is useful for photo tagging.
4. State one limitation and one practical improvement you would investigate.

Do not treat a wrong prediction as a failed activity. It gives us something useful to explain. If your selected label seems sensible, that result alone still does not establish overall model accuracy.

| Item | Your answer |
| --- | --- |
| Question | ... |
| Expected outcome | ... |
| Image description | ... |
| Actual label and score | ... |
| Interpretation | ... |
| Limitation and proposed improvement | ... |

For a real application, define the intended categories, obtain labelled examples from the intended setting, and evaluate the system on separate examples.

# Section 5: Features and CNN intuition (optional extension)

> **Guiding question 5:** What could useful visual features look like?

## A fixed edge filter

Traditional approaches can use a filter specified by a person to highlight changes in intensity. The supplied Pillow `FIND_EDGES` example illustrates one fixed operation. Edge responses can help describe an image, but an edge view by itself does not assign semantic labels.

The next cell is a conceptual illustration. Our MobileNetV2 classifier still receives the RGB image prepared in Section 2; it does not use this edge view as its input.

In [ ]:
gray_img = model_img.convert("L")
edge_img = gray_img.filter(ImageFilter.FIND_EDGES)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(gray_img, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Grayscale image")
axes[1].imshow(edge_img, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Response to a fixed edge filter")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

## Learned feature maps

A CNN learns filter parameters during training. The supplied cell extracts the output of MobileNetV2's first convolution layer and shows six of its feature maps for the current image.

Different channels can respond differently to the same photograph. These maps show internal responses. They are not object masks, attention maps, or a complete explanation of the final prediction. Do not assign a meaning such as "dog detector" to a channel just from looking at it.

Each map uses its own display range so its pattern is visible. Compare **where** responses occur, rather than interpreting relative brightness across panels as a common magnitude scale.

In [ ]:
first_conv = next(layer for layer in model.layers if isinstance(layer, tf.keras.layers.Conv2D))
feature_model = tf.keras.Model(inputs=model.inputs, outputs=first_conv.output)
feature_maps = feature_model.predict(x, verbose=0)[0]
print("Layer:", first_conv.name)
print("Feature map shape (height, width, channels):", feature_maps.shape)

fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for channel, axis in enumerate(axes.flat):
    axis.imshow(feature_maps[:, :, channel], cmap="viridis")
    axis.set_title(f"Feature channel {channel}")
    axis.axis("off")
plt.tight_layout()
plt.show()

## Troubleshooting

| Symptom | Next step |
| --- | --- |
| Message says to complete an exercise | Replace that exercise's `None` with your code and rerun it |
| `NameError` | Run earlier cells in order, especially after restarting the runtime |
| Sample or weights cannot download | Check internet access, retry, or use presenter screenshots |
| Upload cell fails | Choose exactly one JPEG or PNG and rerun the image-loading cell |
| `google.colab` cannot import | Run the upload option in Colab |
| Unexpected batch shape | Check the RGB conversion, target size and batch dimension |
| Prediction seems wrong | Inspect the photo and preprocessing, and consider the model's fixed categories |

After changing an uploaded image, rerun its dependent cells. After changing only `top_k`, rerun the prediction and result-plotting cells.

## References

- [Colab FAQ](https://research.google.com/colaboratory/faq.html): notebook upload and runtime behaviour.
- [TensorFlow MobileNetV2 API](https://www.tensorflow.org/api_docs/python/tf/keras/applications/MobileNetV2): model input, pretrained parameters and categories.
- [MobileNetV2 preprocessing](https://www.tensorflow.org/api_docs/python/tf/keras/applications/mobilenet_v2/preprocess_input): input scaling.
- [Keras Applications](https://keras.io/api/applications/): pretrained image classification workflow.
- [TensorFlow Keras Model API](https://www.tensorflow.org/api_docs/python/tf/keras/Model): extracting intermediate tensors.
- [TensorFlow transfer learning tutorial](https://www.tensorflow.org/tutorials/images/transfer_learning): adaptation to new categories.
- [Pillow image filters](https://pillow.readthedocs.io/en/stable/reference/ImageFilter.html): the supplied fixed edge-filter illustration.
- [TensorFlow sample-image tutorial](https://www.tensorflow.org/tutorials/generative/adversarial_fgsm): source of the Labrador example. This notebook does not implement the tutorial's adversarial attack.
- Sample image: [Wikimedia Commons file page](https://commons.wikimedia.org/wiki/File:YellowLabradorLooking_new.jpg), original Elf photo, derivative/retouch by Djmirko and FT2, [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/).
